# Phase 7: Real Dataset Integration

This notebook downloads the SARS-CoV-2 reference genome `NC_045512.2`, parses FASTA data, generates fixed-length sliding-window fragments, creates adjacent fragment pairs, and benchmarks the existing Hamming Distance CPU and CUDA implementations on real genomic fragments.

If you are not already in the repository root, clone or open the repository first. Example only:

```bash
# !git clone <repository-url>
# %cd CUDA-Bioinformatic
```

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
required_paths = [Path("scripts"), Path("src"), Path("benchmarks")]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(f"This notebook should be run from the repository root. Missing: {missing_paths}")

In [ ]:
!python scripts/download_datasets.py \
  --dataset sars-cov-2 \
  --output data/raw/sars_cov_2_NC_045512_2.fasta

In [ ]:
!python scripts/fragment_fasta.py \
  --input data/raw/sars_cov_2_NC_045512_2.fasta \
  --output-csv data/processed/sars_cov_2_fragments_128.csv \
  --output-txt data/sars_cov_2_fragments_128.txt \
  --output-pairs data/processed/sars_cov_2_pairs_128_stride_32.txt \
  --window-size 128 \
  --stride 32 \
  --skip-ambiguous

In [ ]:
!g++ src/hamming_cpu.cpp -O3 -std=c++17 -I src/common -o hamming_cpu
!nvcc src/hamming_gpu.cu -O3 -std=c++17 -I src/common -o hamming_gpu
!nvcc src/hamming_gpu_encoded.cu -O3 -std=c++17 -I src/common -o hamming_gpu_encoded

In [ ]:
!./hamming_cpu \
  data/processed/sars_cov_2_pairs_128_stride_32.txt \
  results/hamming/real_dataset_hamming_cpu.csv \
  --repetitions 5

!./hamming_gpu \
  data/processed/sars_cov_2_pairs_128_stride_32.txt \
  results/hamming/real_dataset_hamming_gpu_char.csv \
  --repetitions 5

!./hamming_gpu_encoded \
  data/processed/sars_cov_2_pairs_128_stride_32.txt \
  results/hamming/real_dataset_hamming_gpu_encoded.csv \
  --repetitions 5

In [ ]:
!python benchmarks/run_real_dataset_benchmark.py \
  --window-size 128 \
  --stride 32 \
  --repetitions 5

In [ ]:
!python scripts/plot_real_dataset_benchmark.py

In [ ]:
!ls -R data/raw data/processed benchmarks assets/benchmark_charts/real_dataset